#Bidirectional LSTM
Sentiment Analysis


In [3]:
!pip install tensorflow
!pip install numpy pandas matplotlib seaborn scikit-learn

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [7]:
import pandas as pd

# Create a dummy sentiment_data.csv file for demonstration
data = {'text': ["This product is amazing!", "I hate this item, it's terrible.", "It's okay, not great but not bad.", "Absolutely fantastic, love it.", "Worst experience ever, totally disappointed."],
        'sentiment': ["Positive", "Negative", "Neutral", "Positive", "Negative"]}
dummy_df = pd.DataFrame(data)
dummy_df.to_csv('sentiment_data.csv', index=False)

df = pd.read_csv("sentiment_data.csv")
df.head()
texts = df['text'].astype(str)
labels = df['sentiment']

le = LabelEncoder()
labels = le.fit_transform(labels)
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
max_len = 100
X = pad_sequences(sequences, maxlen=max_len)
y = labels

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128, input_length=max_len))
model.add(LSTM(64))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.summary()


history = model.fit(X_train, y_train,
                    epochs=5,
                    batch_size=32,
                    validation_data=(X_test, y_test))

loss, accuracy = model.evaluate(X_test, y_test)
print("Accuracy:", accuracy)

def predict_sentiment(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len)
    prediction = model.predict(padded)

    if prediction[0][0] > 0.5:
        return "Positive"
    else:
        return "Negative"

print(predict_sentiment("This product is amazing"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.2500 - loss: 0.6916 - val_accuracy: 0.0000e+00 - val_loss: 0.7022
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step - accuracy: 0.2500 - loss: 0.6416 - val_accuracy: 0.0000e+00 - val_loss: 0.7196
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - accuracy: 0.2500 - loss: 0.5976 - val_accuracy: 0.0000e+00 - val_loss: 0.7398
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step - accuracy: 0.2500 - loss: 0.5737 - val_accuracy: 0.0000e+00 - val_loss: 0.7643
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - accuracy: 0.2500 - loss: 0.4928 - val_accuracy: 0.0000e+00 - val_loss: 0.7957
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.0000e+00 - loss: 0.7957
Accuracy: 0.0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step
Positive


In [8]:
from tensorflow.keras.layers import Bidirectional

model = Sequential()
model.add(Embedding(5000, 128, input_length=max_len))
model.add(Bidirectional(LSTM(64)))
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


#Sentiment Analysis
Text Auto-Completion

In [12]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from collections import defaultdict, Counter

negative_reviews = [
    "You are so stupid and useless, you can’t do anything right."
"Nobody likes you, just stop talking."
"This is the worst thing I have ever seen, it’s completely trash."
"You are a complete failure and you will never succeed."
"Shut up, your opinion doesn’t matter at all."
"You are disgusting and pathetic."
"I hate you so much, you ruin everything."
"You are an idiot, learn something before speaking."
"This product is terrible, it’s a total waste of money."
"You don’t deserve respect, you are worthless."

]

VOCAB_SIZE = 300

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(negative_reviews)
sequences = tokenizer.texts_to_sequences(negative_reviews)

transitions = defaultdict(list)
for seq in sequences:
    for i in range(len(seq) - 1):
        transitions[seq[i]].append(seq[i + 1])

print("--- Negative comment Sentence Autocompletion ---")

while True:
    partial = input("\nEnter partial sentence (or type 'exit' to quit): ")
    if partial.lower() == 'exit':
        break
    if not partial.strip():
        continue

    seq = tokenizer.texts_to_sequences([partial])[0]
    if not seq:
        print("No recognized words in your input.")
        continue

    last_token = seq[-1]
    candidates = transitions.get(last_token, [])

    if not candidates:
        print("No suggestions available.")
        continue

    count = Counter(candidates)
    top_suggestions = count.most_common(5)

    suggested_words = [tokenizer.index_word.get(idx, "<OOV>") for idx, _ in top_suggestions]

    print(f"\nPartial: '{partial}'")
    print("Suggested next words:")
    for i, word in enumerate(suggested_words, 1):
        print(f"  {i}. {word}")

    best_word = suggested_words[0]
    completed = partial + " " + best_word
    print(f"\nExample completed: '{completed}'")

--- Negative comment Sentence Autocompletion ---

Enter partial sentence (or type 'exit' to quit): worst

Partial: 'worst'
Suggested next words:
  1. thing

Example completed: 'worst thing'

Enter partial sentence (or type 'exit' to quit): exit
